# Follow work across EPSM packages

Use installed coordinated logging packages and the examples extra. This offline example uses synthetic building evidence, explicit teaching assumptions and the weather file packaged with RC. No random sampling or physical-unit conversion is introduced by logging. The expected outcome is a real annual RC result, LCA assessment and retained JSONL trace.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from epsm import configure_logging, reset_logging

workspace = TemporaryDirectory()
output = Path(workspace.name)
configure_logging(directory=output)

INFO shows major stages on stderr; DEBUG is retained in `logs/study.jsonl`. To focus on Core, override its console level. Off stops both destinations. Reset restores the host application settings.


In [ ]:
configure_logging(directory=output, package_levels={"epsm_core": "DEBUG"})
configure_logging(enabled=False)
reset_logging()

Run the example helper through the installed public APIs. It connects a portable database record, a physical Building, RC and LCA, and verifies operation relationships. The helper closes its handlers before returning.


In [ ]:
from logging_walkthrough import run

result = run(output)
assert result.energy.complete

In [ ]:
import json

events = [
    json.loads(line) for line in (output / "logs/study.jsonl").read_text().splitlines()
]
assert any(event["event"] == "evaluation.saved" for event in events)
[(event["logger"], event["event"]) for event in events if event["level"] == "INFO"][-8:]

Inspect `logging_workers.py` for a standard-library queue/listener example. Each process receives its context explicitly; one listener owns file output. Raw engine logs, saved results and diagnostics remain separate. In real work, choose a retained output directory instead of this temporary one.


In [ ]:
reset_logging()
workspace.cleanup()